In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
x = torch.rand(5,3)
print(x)

tensor([[0.3085, 0.3470, 0.8932],
        [0.6775, 0.2536, 0.7483],
        [0.0233, 0.8872, 0.1950],
        [0.9115, 0.4765, 0.4862],
        [0.9959, 0.8468, 0.6813]])


In [3]:
x = torch.tensor([1,2,3])
print(x)

y = torch.tensor([4,5,6])
print(y)

print(x+y)
print(x*y)

tensor([1, 2, 3])
tensor([4, 5, 6])
tensor([5, 7, 9])
tensor([ 4, 10, 18])


In [4]:
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor(2.0, requires_grad=True)    #requires_grad=True このwについて勾配を追跡・計算できるようにする

y = w * x
loss = y.sum()

loss.backward()    #lossをwで微分　w.gradに計算結果（勾配）を保存

print(w.grad)    #gradは勾配を入れておく場所

tensor(6.)


In [5]:
w = torch.tensor(2.0, requires_grad=True)

x = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([4.0, 8.0, 12.0])

optimizer = torch.optim.SGD([w], lr=0.02)    #optim.SGDはパラメータwの更新方法 Irは学習率　勾配をどれだけ使って重みを動かすか決める
#SGD = Stochastic Gradient Descent（確率的勾配降下法）

for i in range(10):
    y = w * x

    loss = ((y - target) ** 2).sum()

    optimizer.zero_grad()    #勾配を消す　これがないと勾配が蓄積していく
    loss.backward()    #新しい勾配を計算
    optimizer.step()    #wを更新　optimizerを実行

    print(i, "w =", w.item(), "loss =", loss.item())    #.itemは1要素だけ入っている値をPythonの数値として表示

0 w = 3.119999885559082 loss = 56.0
1 w = 3.612799882888794 loss = 10.841602325439453
2 w = 3.829631805419922 loss = 2.0989344120025635
3 w = 3.9250380992889404 loss = 0.40635451674461365
4 w = 3.9670166969299316 loss = 0.07867012172937393
5 w = 3.985487222671509 loss = 0.015230482444167137
6 w = 3.993614435195923 loss = 0.0029486692510545254
7 w = 3.997190237045288 loss = 0.0005708470125682652
8 w = 3.9987637996673584 loss = 0.00011053076741518453
9 w = 3.9994561672210693 loss = 2.139644493581727e-05


In [6]:
class Mymodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1,1)    #入力1個→出力1個の全結合層を作る
        #y = ωx + bを行う   ωとbは学習可能なパラメータ

    def forward(self,x):
        return self.linear(x)

model = Mymodel()

print(model)


Mymodel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
x = torch.tensor([[1.0],[2.0],[3.0],[4.0]])
target = torch.tensor([[2.0],[4.0],[6.0],[8.0]])
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.02
)

for i in range(1000):
    y = model(x)    #予測

    loss = ((y -target)**2).mean()    #誤差

    optimizer.zero_grad()    #勾配のリセット

    loss.backward()    #勾配計算

    optimizer.step()    #パラメータ更新

print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[1.9989]], requires_grad=True)
Parameter containing:
tensor([0.0031], requires_grad=True)


In [8]:
class MyDataset(Dataset):
    def __init__(self,x,target):
        self.x = x
        self.target = target

    def __len__(self):    #データの数を数える
        return len(self.x)

    def __getitem__(self,index):    #指定されたデータ（〇番目）を返す
        return self.x[index], self.target[index]

In [9]:
dataset = MyDataset()

loader = DataLoader(
    dataset,    #データを取るデータセット
    batch_size=2,    #データを2個ずつ
    shuffle=True    #エポックごとにデータの順番をシャッフル
)

TypeError: MyDataset.__init__() missing 2 required positional arguments: 'x' and 'target'

In [10]:
for x, target in loader:
    print(x)
    print(target)

NameError: name 'loader' is not defined

In [11]:
model = Mymodel()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

for epoch in range(10):    #全データを1回参照するのが1エポック　バッチサイズが2、データ総数が4なので2データずつの2stepで1epoch
#エポックがデータを何周するか、ステップは1バッチを使ってパラメータを1回更新
    
    for x, target in loader:    #バッチ数だけ繰り返す　x,targetをloaderから受け取る
                                #バッチ数=データ総数/バッチサイズ　drop_last=TrueをDataloaderの引数に追加すると切り捨てで処理　初期値は切り上げ
        y = model(x)

        loss = ((y - target) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(
        "epoch:", epoch,
        "loss:", loss.item()
    )

NameError: name 'dataset' is not defined

In [12]:
model = nn.Sequential(    #書いた順にモジュールをつないで1つのモジュールとして使えるようにするモジュール
    nn.Linear(1,10),
    nn.ReLU(),    #入力に対して―は0に、＋はそのままで返すReLU関数
    nn.Linear(10,1)
)

print(model)

Sequential(
  (0): Linear(in_features=1, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
)


In [13]:
class Mymodel_MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.netwark = nn.Sequential(    #ここがMLP(Multi-Layer Perceptron) 
            nn.Linear(1,10),
            nn.ReLU(),
            nn.Linear(10,10),
            nn.ReLU(),
            nn.Linear(10,1)
        )

    def forward(self,x):    #作った処理を実行するためのもの　forwardの中で使う順番を決められる
        return self.netwark(x)

In [14]:
model = Mymodel_MLP()

x = torch.tensor([[1.0],[2.0],[3.0],[4.0]])
target = torch.tensor([[1.0],[4.0],[9.0],[16.0]])

dataset = MyDataset()

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

for epoch in range(100):

    total_loss = 0
    
    for x, target in loader:

        y = model(x)
        loss = ((y - target)**2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    print(epoch, average_loss)

TypeError: MyDataset.__init__() missing 2 required positional arguments: 'x' and 'target'

In [15]:
x = torch.arange(1, 11, dtype=torch.float32).reshape(-1, 1)
target = x * 2

train_x = x[:8]
train_target = target[:8]

test_x = x[8:]
test_target = target[8:]

model = Mymodel_MLP()

train_dataset = MyDataset(train_x, train_target)

loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.005
)

for epoch in range(100):

    for x, target in loader:

        y = model(x)
        loss = ((y - target)**2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

with torch.no_grad():
    train_prediction = model(train_x)
    test_prediction = model(test_x)

print("train prediction:")
print(train_prediction)

print("train target:")
print(train_target)

print("test prediction:")
print(test_prediction)

print("test target:")
print(test_target)

train prediction:
tensor([[ 2.0058],
        [ 3.9973],
        [ 5.9964],
        [ 7.9955],
        [ 9.9947],
        [11.9938],
        [13.9929],
        [15.9921]])
train target:
tensor([[ 2.],
        [ 4.],
        [ 6.],
        [ 8.],
        [10.],
        [12.],
        [14.],
        [16.]])
test prediction:
tensor([[17.9669],
        [19.9419]])
test target:
tensor([[18.],
        [20.]])


In [16]:
x = torch.arange(1, 11, dtype=torch.float32).reshape(-1, 1)
target = 2 * x

train_x = x[:8]
train_target = target[:8]

test_x = x[8:]
test_target = target[8:]

train_dataset = MyDataset(train_x, train_target)

loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

model = nn.Sequential(
    nn.Linear(1, 10),
    nn.Linear(10, 10),
    nn.Linear(10, 1)
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0005
)

for epoch in range(100):

    for x, target in loader:

        y = model(x)
        loss = ((y - target)**2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

with torch.no_grad():
    train_prediction = model(train_x)

print("train prediction:")
print(train_prediction)

print("train target:")
print(train_target)

print("test prediction:")
print(test_prediction)

print("test target:")
print(test_target)

train prediction:
tensor([[ 2.2901],
        [ 4.2300],
        [ 6.1698],
        [ 8.1097],
        [10.0496],
        [11.9894],
        [13.9293],
        [15.8692]])
train target:
tensor([[ 2.],
        [ 4.],
        [ 6.],
        [ 8.],
        [10.],
        [12.],
        [14.],
        [16.]])
test prediction:
tensor([[17.9669],
        [19.9419]])
test target:
tensor([[18.],
        [20.]])


In [17]:
x = torch.arange(1, 11, dtype=torch.float32).reshape(-1, 1)
target = x ** 3

train_x = x[:8]
train_target = target[:8]

test_x = x[8:]
test_target = target[8:]

model = Mymodel_MLP()

train_dataset = MyDataset(train_x, train_target)

loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.00005
)

for epoch in range(100):

    for x, target in loader:

        y = model(x)
        loss = ((y - target)**2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

with torch.no_grad():
    train_prediction = model(train_x)
    test_prediction = model(test_x)

print("ReLU関数あり")
print("train prediction:")
print(train_prediction)

print("train target:")
print(train_target)

ReLU関数あり
train prediction:
tensor([[  0.5466],
        [  0.4982],
        [ 45.1423],
        [120.7550],
        [196.3678],
        [271.9805],
        [347.5933],
        [423.0309]])
train target:
tensor([[  1.],
        [  8.],
        [ 27.],
        [ 64.],
        [125.],
        [216.],
        [343.],
        [512.]])


In [18]:
x = torch.arange(1, 11, dtype=torch.float32).reshape(-1, 1)
target = x ** 3

train_x = x[:8]
train_target = target[:8]

test_x = x[8:]
test_target = target[8:]

model = nn.Sequential(
    nn.Linear(1, 10),
    nn.Linear(10, 10),
    nn.Linear(10, 1)
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.00005
)

for epoch in range(100):

    for x, target in loader:

        y = model(x)
        loss = ((y - target)**2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

with torch.no_grad():
    train_prediction = model(train_x)

print("train prediction:")
print(train_prediction)

print("train target:")
print(train_target)

train prediction:
tensor([[-57.8118],
        [ -7.8290],
        [ 42.1537],
        [ 92.1365],
        [142.1192],
        [192.1020],
        [242.0848],
        [292.0675]])
train target:
tensor([[  1.],
        [  8.],
        [ 27.],
        [ 64.],
        [125.],
        [216.],
        [343.],
        [512.]])


In [19]:
x = torch.tensor(
    [-2.0, -1.0, 2.0, 3.0],
    requires_grad=True
)

y = torch.relu(x)

loss = y.sum()

loss.backward()

print("x =", x)
print("y =", y)
print("gradient =", x.grad)

x = tensor([-2., -1.,  2.,  3.], requires_grad=True)
y = tensor([0., 0., 2., 3.], grad_fn=<ReluBackward0>)
gradient = tensor([0., 0., 1., 1.])
